# Task 1: Data Preprocessing and Exploration

## Objective
Load, clean, and understand the data to prepare it for modeling.

## Setup and Imports

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from scipy import stats
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Import custom modules
import sys
sys.path.append('..')
from src.data_loader import DataLoader
from src.preprocess import DataPreprocessor
from src.models import TimeSeriesModels

ModuleNotFoundError: No module named 'pandas'

## 1. Load and Explore Data

In [ ]:
# Load data from CSV files
def load_data(data_dir="data/processed"):
    tickers = ['TSLA', 'BND', 'SPY']
    data = {}
    for ticker in tickers:
        filepath = os.path.join(data_dir, f"{ticker}_data.csv")
        if os.path.exists(filepath):
            df = pd.read_csv(filepath, parse_dates=['Date'])
            df.set_index('Date', inplace=True)
            data[ticker] = df
            print(f"✓ Loaded {ticker}: {len(df)} rows")
            print(f"  Period: {df.index.min()} to {df.index.max()}")
    return data

data = load_data()

# Display sample
data['TSLA'].head()

In [ ]:
# Basic statistics
for ticker, df in data.items():
    print(f"\n{ticker} Statistics:")
    print("="*50)
    print(df[['Open', 'High', 'Low', 'Close', 'Volume']].describe())

## 2. Data Cleaning and Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor()

# Process each asset
processed_data = {}

for ticker, df in data.items():
    print(f"\nProcessing {ticker}...")
    
    # Check for missing values
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f"  Missing values: {missing[missing > 0].to_dict()}")
        df = preprocessor.handle_missing_values(df, method='forward_fill')
        print(f"  Missing values handled")
    
    # Calculate returns
    df = preprocessor.calculate_returns(df)
    
    # Calculate rolling volatility
    df = preprocessor.calculate_volatility(df, window=30)
    
    processed_data[ticker] = df
    print(f"  ✓ Preprocessing complete")

## 3. Exploratory Data Analysis

### 3.1 Closing Price Trends

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(3, 1, figsize=(15, 12))
fig.suptitle('Historical Closing Prices (2015-2026)', fontsize=16, fontweight='bold')

for idx, (ticker, df) in enumerate(processed_data.items()):
    ax = axes[idx]
    ax.plot(df.index, df['Close'], label=ticker, linewidth=2)
    ax.set_title(f'{ticker} - Closing Price')
    ax.set_xlabel('Date')
    ax.set_ylabel('Price ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 Daily Returns Distribution

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12))
fig.suptitle('Daily Returns Distribution', fontsize=16, fontweight='bold')

for idx, (ticker, df) in enumerate(processed_data.items()):
    ax = axes[idx]
    df['Daily_Return'].hist(bins=50, alpha=0.7, edgecolor='black', linewidth=1, ax=ax)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
    ax.axvline(x=df['Daily_Return'].mean(), color='green', linestyle='-', linewidth=2, label='Mean')
    ax.set_title(f'{ticker} - Daily Returns')
    ax.set_xlabel('Daily Return')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add statistics
    mean_return = df['Daily_Return'].mean()
    std_return = df['Daily_Return'].std()
    skew = df['Daily_Return'].skew()
    kurt = df['Daily_Return'].kurtosis()
    ax.text(0.95, 0.95, f'Mean: {mean_return:.6f}\nStd: {std_return:.6f}\nSkew: {skew:.3f}\nKurt: {kurt:.3f}',
            transform=ax.transAxes, verticalalignment='top',
            horizontalalignment='right', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

### 3.3 Rolling Volatility

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12))
fig.suptitle('Rolling Volatility (30-day window)', fontsize=16, fontweight='bold')

for idx, (ticker, df) in enumerate(processed_data.items()):
    ax = axes[idx]
    ax.plot(df.index, df['Rolling_Volatility'], label='30-day Volatility', linewidth=2)
    ax.set_title(f'{ticker}')
    ax.set_xlabel('Date')
    ax.set_ylabel('Volatility')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.4 Outlier Detection

In [ ]:
# Detect outliers in returns
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Outlier Detection in Daily Returns', fontsize=16, fontweight='bold')

for idx, (ticker, df) in enumerate(processed_data.items()):
    ax = axes[idx]
    
    # Detect outliers using IQR method
    outliers = preprocessor.detect_outliers(df, 'Daily_Return', method='iqr')
    
    # Plot returns with outliers highlighted
    ax.plot(df.index, df['Daily_Return'], 'o', markersize=2, alpha=0.6, label='Normal')
    ax.plot(df.index[outliers], df.loc[outliers, 'Daily_Return'], 'ro', markersize=8, label='Outliers')
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    ax.set_title(f'{ticker} - {outliers.sum()} outliers')
    ax.set_xlabel('Date')
    ax.set_ylabel('Daily Return')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Stationarity Tests

Testing for stationarity using Augmented Dickey-Fuller test.

In [ ]:
from statsmodels.tsa.stattools import adfuller

stationarity_results = {}

for ticker, df in processed_data.items():
    print(f"\n{ticker} Stationarity Test:")
    print("="*50)
    
    # Test closing prices
    result_price = adfuller(df['Close'].dropna())
    print(f"  Closing Price:")
    print(f"    ADF Statistic: {result_price[0]:.4f}")
    print(f"    p-value: {result_price[1]:.6f}")
    print(f"    Critical Values: {result_price[4]}")
    print(f"    Stationary: {'Yes' if result_price[1] < 0.05 else 'No'}")
    
    # Test daily returns
    result_returns = adfuller(df['Daily_Return'].dropna())
    print(f"\n  Daily Returns:")
    print(f"    ADF Statistic: {result_returns[0]:.4f}")
    print(f"    p-value: {result_returns[1]:.6f}")
    print(f"    Critical Values: {result_returns[4]}")
    print(f"    Stationary: {'Yes' if result_returns[1] < 0.05 else 'No'}")
    
    stationarity_results[ticker] = {
        'price': result_price,
        'returns': result_returns
    }

## 5. Risk Metrics

### 5.1 Value at Risk (VaR)

In [ ]:
def calculate_var(returns, confidence_level=0.95):
    """Calculate Value at Risk."""
    var_parametric = returns.mean() - returns.std() * stats.norm.ppf(confidence_level)
    var_historical = np.percentile(returns, (1 - confidence_level) * 100)
    return var_parametric, var_historical

# Calculate VaR for each asset
for ticker, df in processed_data.items():
    returns = df['Daily_Return'].dropna()
    var_parametric, var_historical = calculate_var(returns)
    
    print(f"\n{ticker} Value at Risk (95% confidence):")
    print("="*50)
    print(f"  Parametric VaR: {var_parametric:.4f} ({var_parametric*100:.2f}%)")
    print(f"  Historical VaR: {var_historical:.4f} ({var_historical*100:.2f}%)")

### 5.2 Sharpe Ratio

In [ ]:
def calculate_sharpe(returns, risk_free_rate=0.02):
    """Calculate Sharpe ratio."""
    excess_returns = returns - risk_free_rate / 252
    sharpe = np.sqrt(252) * excess_returns.mean() / returns.std()
    return sharpe

# Calculate Sharpe ratio for each asset
for ticker, df in processed_data.items():
    returns = df['Daily_Return'].dropna()
    sharpe = calculate_sharpe(returns)
    print(f"{ticker} Sharpe Ratio: {sharpe:.4f}")

## 6. Summary and Key Insights

### Key Findings:

In [ ]:
# Summary statistics table
summary_df = pd.DataFrame()

for ticker, df in processed_data.items():
    returns = df['Daily_Return'].dropna()
    summary_df[ticker] = [
        len(df),
        df['Close'].iloc[-1],
        df['Close'].min(),
        df['Close'].max(),
        returns.mean(),
        returns.std(),
        returns.skew(),
        returns.kurtosis(),
        calculate_sharpe(returns)
    ]

summary_df.index = ['Rows', 'Latest Price', 'Min Price', 'Max Price', 
                    'Mean Return', 'Std Return', 'Skewness', 'Kurtosis', 'Sharpe Ratio']

print("Summary Statistics:")
print("="*60)
print(summary_df.round(4))

### Key Insights:

1. **TSLA** shows the highest volatility and potential returns, consistent with its high-risk profile
2. **BND** demonstrates stability with low volatility, suitable for risk-averse investors
3. **SPY** provides balanced exposure with moderate risk and returns
4. **Stationarity**: All assets show non-stationary prices but stationary returns (p < 0.05), validating the need for differencing in ARIMA models
5. **Outliers**: TSLA has the most outliers, indicating higher frequency of extreme moves
6. **Risk Metrics**: TSLA has the highest VaR and Sharpe ratio, suggesting higher risk-adjusted returns

## Next Steps

Proceed to Task 2: Build Time Series Forecasting Models